# Base line

In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score, 
    recall_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

2025-06-17 22:01:20.833135: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-17 22:01:20.833275: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-17 22:01:20.846983: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-17 22:01:21.820479: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Cargar dataset

In [2]:
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx'
test_file_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx'

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")

Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


### Pre-procesamiento

In [3]:
# --- AJUSTE 3: Crear el Input Estructurado ---
# print("Creando input estructurado...")
# df_train['structured_text'] = df_train.apply(
#     lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
#     axis=1
# )

# Renombrar columnas para el modelo
# df_model = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})

df_model = df_train.rename(columns={'Review': 'text', 'Polarity': 'label'})

df_model['label'] = df_model['label'].apply(lambda x: int(x) - 1)

### Carga del modelo checkpoint

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint = 'distilbert/distilbert-base-multilingual-cased'
model_name = model_checkpoint.replace('/','_')

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=5
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Train/Test split y Tokenizacion

In [5]:
# Convertir a Dataset
dataset = Dataset.from_pandas(df_model)

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Fine tunning

In [6]:
# --- Metricas de evaluacion --- 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.array(labels)
    predictions = np.array(predictions)

    if len(labels) == 0:
        return {
            "accuracy": 0.0,
            "f1_weighted_sklearn": 0.0,
            "precision_macro": 0.0,
            "recall_macro": 0.0,
            "f1_macro": 0.0,
        }

    # --- Métricas conocidas ---
    standard_accuracy = accuracy_score(labels, predictions)
    standard_f1_weighted = f1_score(labels, predictions, average="weighted", zero_division=0)

    # Métricas con promedio 'macro' (trata todas las clases por igual)
    precision_macro = precision_score(labels, predictions, average="macro", zero_division=0)
    recall_macro = recall_score(labels, predictions, average="macro", zero_division=0)
    f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)

    return {
        "accuracy": standard_accuracy,
        "f1_weighted_sklearn": standard_f1_weighted,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }

# --- Configuración de los Argumentos de Entrenamiento ---
training_args = TrainingArguments(
    output_dir=f"./results_{model_name}",        # Directorio para guardar los resultados
    num_train_epochs=2,                          # Número de pasadas completas por el dataset
    per_device_train_batch_size=16,              # Tamaño del lote por GPU/CPU para entrenamiento
    per_device_eval_batch_size=16,               # Tamaño del lote por GPU/CPU para evaluación
    warmup_steps=50,                             # Número de pasos para calentamiento
    weight_decay=0.01,                           # Regularización L2 para evitar sobreajuste
    logging_dir=f'./logs_{model_name}',          # Directorio para los logs (usar con TensorBoard)
    logging_steps=100,                           # Registrar métricas cada X pasos
    eval_strategy="epoch",                       # Evaluar en el conjunto de validación al final de cada época
    save_strategy="epoch",                       # Guardar el modelo al final de cada época
    load_best_model_at_end=True,                 # Cargar el mejor modelo (según metric_for_best_model) al finalizar
    greater_is_better=True,                      # Indica que un valor más alto de la metrica es mejor
    learning_rate=2e-5,                          # Tasa de aprendizaje
    report_to="tensorboard",                     # Permite ver el progreso en TensorBoard
)

# --- Inicializar el Trainer ---
trainer = Trainer(
    model=model,                            
    args=training_args,                     
    train_dataset=tokenized_train_dataset, 
    eval_dataset=tokenized_eval_dataset,   
    compute_metrics=compute_metrics,        
)

# --- Iniciar el Entrenamiento ---
print("--- Iniciando el Fine-tuning con Trainer ---")
trainer.train()
print("--- Fine-tuning Completado ---")
print(f"El mejor modelo se ha guardado en: {trainer.state.best_model_checkpoint}")
# Después de trainer.train(), el mejor modelo se carga automáticamente en 'model'

--- Iniciando el Fine-tuning con Trainer ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted Sklearn,Precision Macro,Recall Macro,F1 Macro
1,1.358400,1.309070,0.420000,0.367779,0.426178,0.390872,0.363748
2,1.117200,1.110480,0.518000,0.508790,0.507719,0.508785,0.501648


--- Fine-tuning Completado ---
El mejor modelo se ha guardado en: ./results_distilbert_distilbert-base-multilingual-cased/checkpoint-282
